# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR² dataset using the `mlcroissant` library, employing the Croissant metadata standard and referencing all major entities by their `@id` values.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the croissant dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset loaded: {getattr(metadata, 'name', None)}")
print(f"Description: {getattr(metadata, 'description', None)}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s. All references to Croissant entities use the `@id` field for maximum reproducibility.

In [ ]:
# Helper: Show all record sets and field IDs
record_sets = list(dataset.metadata.record_sets)

print("Available Record Sets and their fields:")
record_set_ids = []
for rs in record_sets:
    print(f'Record set: {rs.id}')
    record_set_ids.append(rs.id)
    print(f"  Name: {rs.name}")
    print(f"  Description: {rs.description or '(No description)'}")
    print(f"  Fields:")
    for f in rs.fields:
        print(f"   - {f.id}: {f.name} ({f.data_type or 'unknown'})")
    print()

# List field IDs for a specific records set with fields
# For the example dataset, we will attempt to use the first record set
if len(record_sets) == 0:
    print('No record sets found in this dataset. Check schema or Croissant version.')

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview for reproducibility and clarity.

In [ ]:
dfs = {}
for rs in record_sets:
    # Collect by @id for clarity
    print(f"Loading record set: {rs.id} -> {rs.name}")
    try:
        # Use records(record_set=...) with proper @id
        records = list(dataset.records(record_set=rs.id))
        if len(records) > 0:
            dfs[rs.id] = pd.DataFrame(records)
            print(f"  [Loaded {len(dfs[rs.id])} records and {len(dfs[rs.id].columns)} columns.]")
        else:
            print('  [No records found.]')
    except Exception as e:
        print(f"  [Error: {e}]")
if dfs:
    # Preview one loaded DataFrame
    example_key = list(dfs.keys())[0]
    print(f"\nExample columns for {example_key}:")
    print(dfs[example_key].columns.tolist())
    display(dfs[example_key].head())
else:
    print('No DataFrames loaded. Check schema structure.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. All operations reference the appropriate field/column by their `@id` wherever possible.

Note: Be sure to replace the variable assignments below based on the actual field `@id` from your prior output cell.

In [ ]:
# Variables (update these with actual IDs from above)
example_record_set = list(dfs.keys())[0] if dfs else None
df = dfs[example_record_set] if dfs else None

# Attempt to choose a numeric field by Croissant `@id` (else fallback)
numeric_field_id = None
group_field_id = None
if example_record_set:
    # Find likely numeric column
    rs_obj = [rs for rs in record_sets if rs.id == example_record_set][0]
    numeric_fields = [f for f in rs_obj.fields if f.data_type in ('schema:Float', 'schema:Integer', 'schema:Number')]
    if numeric_fields:
        numeric_field_id = numeric_fields[0].id
    # Attempt groupable field (string/categorical)
    group_fields = [f for f in rs_obj.fields if f.data_type == 'schema:Text']
    if group_fields:
        group_field_id = group_fields[0].id

if df is not None and numeric_field_id is not None:
    # Filter on numeric field by threshold
    # Assumes the field is present in the DataFrame
    field_col = numeric_field_id
    threshold = df[field_col].mean() if pd.api.types.is_numeric_dtype(df[field_col]) else 10
    # Keep only rows with numeric_field_id > threshold
    filtered_df = df[df[field_col] > threshold]
    print(f"Filtered records with {field_col} > {threshold}")
    display(filtered_df.head())
    # Normalize
    norm_col = f"{field_col}_normalized"
    filtered_df[norm_col] = (filtered_df[field_col] - filtered_df[field_col].mean()) / filtered_df[field_col].std()
    print(f"Normalized {field_col} for filtered records:")
    display(filtered_df[[field_col, norm_col]].head())
    # Group by (if suitable group field found)
    if group_field_id and group_field_id in filtered_df.columns:
        grouped = filtered_df.groupby(group_field_id)[field_col].mean()
        print(f"Grouped mean of {field_col} by {group_field_id}:")
        display(grouped.head())
else:
    print("No numeric field found or no data loaded. Check dataset structure and Croissant field definitions.")

## 5. Visualization
Visualize data distributions or relationships between fields. You can adapt the following code to visualize, for example, histogram distributions for a numeric field or bar plots grouped by a categorical field.
All axes use Croissant `@id` values for clarity and reproducibility.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and numeric_field_id is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id], bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    # If group_field exists, show a boxplot
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()
else:
    print('No numerical or groupable field available for visualization.')

## 6. Conclusion
In this notebook, we have loaded and explored the FAIR² dataset using the `mlcroissant` library and the Croissant metadata. All steps referenced dataset entities by their `@id` for reproducibility. You can now extend this notebook with further analyses (e.g., machine learning, advanced visualizations) tailored to your research or application needs.